# Dynamic Carbon Cycle Model

A six-pool dynamic carbon cycle model simulating how human emissions have altered the natural carbon cycle and exploring future emission scenarios to 2100.

**Pools modelled:** Atmosphere · Biosphere · Soils · Shallow Ocean · Deep Ocean · Geosphere

**Methods:** Piecewise-linear emission functions · ODE integration (Runge-Kutta) · BAU vs mitigation scenario comparison · Sensitivity analysis

## 1. Emission Functions

Historical emission rates modelled as piecewise-linear functions (PgC yr⁻¹).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

def land_use_emissions(t):
    """Land-use carbon emissions (PgC/yr) as a function of year."""
    if t < 1750:
        return 0
    elif t < 1900:
        return 0.0033 * (t - 1750)          # Linear growth phase 1
    elif t < 1950:
        return 0.495 + 0.01 * (t - 1900)    # Linear growth phase 2
    else:
        return 0.995                         # Stabilised from 1950

def fossil_emissions(t):
    """Fossil fuel carbon emissions (PgC/yr) as a function of year."""
    if t < 1850:
        return 0
    elif t < 1950:
        return 0.01483 * (t - 1850)          # Pre-industrial growth
    else:
        return 1.483 + 0.1217 * (t - 1950)  # Accelerated post-1950 growth

In [ ]:
# Plot historical emission trajectories
years = np.arange(1750, 2026)
land_emis = [land_use_emissions(y) for y in years]
fossil_emis = [fossil_emissions(y) for y in years]

plt.figure(figsize=(12, 6))
plt.plot(years, land_emis, label='Land-use Emissions', linewidth=2, color='green')
plt.plot(years, fossil_emis, label='Fossil Fuel Emissions', linewidth=2, color='red')
plt.axvline(x=1900, color='gray', linestyle=':', alpha=0.7)
plt.axvline(x=1950, color='gray', linestyle=':', alpha=0.7)
plt.xlabel('Year')
plt.ylabel('Emissions (PgC yr⁻¹)')
plt.title('Historical Carbon Emissions: Land-use vs Fossil Fuels')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(1750, 2025)
plt.ylim(bottom=0)
plt.tight_layout()
plt.show()

## 2. Six-Pool Carbon Model

Carbon pools (PgC) and their interactions modelled as first-order flux processes.

**Biospheric uptake (photosynthesis):**
$$f = f_0 \left(1 + \beta \ln\frac{A_t}{A_0}\right)$$

**Ocean outgassing:**
$$\psi = k_\psi \left(S_0 + \varepsilon(S_t - S_0)\right), \quad \varepsilon = 3.09 + 1.86\times10^{-2}A^*_t - 1.8\times10^{-6}(A^*_t)^2$$

In [ ]:
def carbon_model_derivatives(current_state, initial_state, beta, f0, k_psi):
    """Compute derivatives for all six carbon pools.
    
    Pools: Atmosphere (A), Biosphere (B), Soils (S),
           Deep Ocean (D), Geosphere (G), Shallow Ocean (So)
    """
    A, B, S, D, G, So = current_state
    A0, B0, S0, D0, G0, So0 = initial_state

    # Atmospheric CO₂ in ppmv (1 ppmv = 2.13 PgC)
    A_ppmv = A / 2.13
    epsilon = 3.09 + 1.86e-2 * A_ppmv - 1.8e-6 * A_ppmv**2

    # Variable fluxes
    f = f0 * (1 + beta * np.log(A / A0))       # Biospheric uptake
    psi = k_psi * (S0 + epsilon * (S - S0))    # Ocean outgassing

    # First-order rate constants
    delta = 62
    k_B_S  = delta / B0
    k_S_A  = delta / S0
    k_A_So = 70 / A0
    k_So_D = 100 / So0
    k_D_So = 100 / D0

    # Pool derivatives
    dA_dt  = -f + delta - k_A_So * A + psi
    dB_dt  = f - 2 * delta
    dS_dt  = 2 * delta - k_S_A * S
    dSo_dt = k_A_So * A - psi + k_D_So * D - k_So_D * So
    dD_dt  = k_So_D * So - k_D_So * D
    dG_dt  = 0

    return [dA_dt, dB_dt, dS_dt, dD_dt, dG_dt, dSo_dt]

In [ ]:
# Model parameters
beta, f0, k_psi = 0.4, 62, 0.07

# Pre-industrial initial pool sizes (PgC)
y0 = [600, 730, 1250, 40000, 90000000, 1000]

# Solve over 200 years with no anthropogenic emissions
t = np.arange(0, 201)

def model_dydt(y, t, y0, beta, f0, k_psi):
    return carbon_model_derivatives(y, y0, beta, f0, k_psi)

solution = odeint(model_dydt, y0, t, args=(y0, beta, f0, k_psi))
A_t, B_t, S_t, D_t, G_t, So_t = solution.T

print("Model solved. Final atmospheric carbon:", round(A_t[-1], 2), "PgC")

In [ ]:
# Carbon pool evolution — natural cycle (no anthropogenic emissions)
pools = {
    'Atmosphere': 608, 'Biosphere': 730, 'Soils': 1250,
    'Shallow Ocean': 1000, 'Deep Ocean': 40000, 'Geosphere': 90000000
}

years_plot = np.arange(1750, 2026)
fig, ax1 = plt.subplots(figsize=(12, 8))
ax2 = ax1.twinx()

# Smaller pools on primary axis
for name, color in zip(['Atmosphere', 'Biosphere', 'Soils', 'Shallow Ocean'],
                       ['red', 'green', 'brown', 'blue']):
    ax1.plot(years_plot, np.full_like(years_plot, pools[name]), color=color, linewidth=2, label=name)

# Larger pools scaled on secondary axis
ax2.plot(years_plot, np.full_like(years_plot, pools['Deep Ocean'] / 100),
         'darkblue', linewidth=2, linestyle='--', label='Deep Ocean (×10⁻²)')
ax2.plot(years_plot, np.full_like(years_plot, pools['Geosphere'] / 100000),
         'purple', linewidth=2, linestyle=':', label='Geosphere (×10⁻⁵)')

ax1.set(xlabel='Year', ylabel='Carbon (PgC) — Smaller Pools',
        xlim=(1750, 2025), ylim=(500, 1600))
ax2.set_ylabel('Carbon (PgC) — Larger Pools')
ax1.grid(True, alpha=0.3)
fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.9), fontsize=10)
plt.title('Carbon Pool Sizes — Natural Cycle (No Anthropogenic Emissions)')
plt.tight_layout()
plt.show()

## 3. Anthropogenic Emissions Scenarios (1750–2100)

Three scenarios compared:
- **Business-as-Usual (BAU):** emissions continue on historical trajectory
- **Moderate Mitigation:** 50% reduction by 2050, then stabilised
- **Aggressive Mitigation:** 80% reduction by 2050

In [ ]:
years_full = np.arange(1750, 2101)
A0, B0, S0, Os0, Od0 = 608, 730, 1250, 1000, 40000

def run_scenario(mitigation_factor=1.0):
    """Run carbon model with optional mitigation factor applied to fossil emissions post-2025."""
    A, B, S, O_s, O_d = [np.full_like(years_full, val, dtype=float)
                          for val in [A0, B0, S0, Os0, Od0]]

    for i in range(1, len(years_full)):
        year = years_full[i]

        delta_emis = land_use_emissions(year)
        gamma_base = fossil_emissions(year)

        # Apply mitigation from 2025 onward
        if year > 2025:
            gamma_emis = gamma_base * mitigation_factor
        else:
            gamma_emis = gamma_base

        ocean_uptake = max(0.025 * (A[i-1] - 600), 0)
        land_uptake  = max(0.020 * (A[i-1] - 600), 0)
        ocean_mixing = 0.015 * (O_s[i-1] - O_d[i-1] / 40)

        A[i]   = A[i-1] + delta_emis + gamma_emis - ocean_uptake - land_uptake
        O_s[i] = O_s[i-1] + ocean_uptake - ocean_mixing
        O_d[i] = O_d[i-1] + ocean_mixing
        B[i]   = B[i-1] + 0.6 * land_uptake
        S[i]   = S[i-1] + 0.4 * land_uptake

    return A, B, S, O_s, O_d

# Run all three scenarios
A_bau,  *_ = run_scenario(mitigation_factor=1.0)    # Business-as-usual
A_mod,  *_ = run_scenario(mitigation_factor=0.5)    # Moderate mitigation
A_agg,  *_ = run_scenario(mitigation_factor=0.2)    # Aggressive mitigation

print(f"BAU atmospheric CO₂ in 2100:                {A_bau[-1]/2.13:.0f} ppm")
print(f"Moderate mitigation CO₂ in 2100:            {A_mod[-1]/2.13:.0f} ppm")
print(f"Aggressive mitigation CO₂ in 2100:          {A_agg[-1]/2.13:.0f} ppm")

In [ ]:
# Scenario comparison plot
plt.figure(figsize=(12, 6))
plt.plot(years_full, A_bau / 2.13, 'red',   linewidth=2, label='Business-as-Usual')
plt.plot(years_full, A_mod / 2.13, 'orange', linewidth=2, label='Moderate Mitigation (50% cut by 2050)')
plt.plot(years_full, A_agg / 2.13, 'green',  linewidth=2, label='Aggressive Mitigation (80% cut by 2050)')
plt.axvline(x=2025, color='gray', linestyle=':', alpha=0.7, label='Mitigation start (2025)')
plt.xlabel('Year')
plt.ylabel('Atmospheric CO₂ (ppm)')
plt.title('Atmospheric CO₂ Under Different Emission Scenarios (1750–2100)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Target Scenario: Return to 300 ppm by 2100

Binary search to find the required annual fossil emission reduction rate (PgC/yr per year) from 2025.

In [ ]:
target_pgc   = 300 * 2.13   # 300 ppm in PgC
tolerance    = 0.1 * 2.13   # ±0.1 ppm tolerance

def run_model_with_reduction(annual_reduction):
    """Run model where fossil emissions decline linearly from 2025 at given annual rate."""
    A, B, S, O_s, O_d = [np.full_like(years_full, val, dtype=float)
                          for val in [A0, B0, S0, Os0, Od0]]
    baseline_2025 = fossil_emissions(2025)

    for i in range(1, len(years_full)):
        year = years_full[i]
        delta_emis = land_use_emissions(year)

        if year < 2025:
            gamma_emis = fossil_emissions(year)
        else:
            gamma_emis = max(baseline_2025 + annual_reduction * (year - 2025), 0)

        ocean_uptake = max(0.033 * (A[i-1] - 600), 0)
        land_uptake  = max(0.028 * (A[i-1] - 600), 0) * (1 + 0.18 * np.log(A[i-1] / A0))
        ocean_mixing = 0.018 * (O_s[i-1] - O_d[i-1] / 40)

        A[i]   = A[i-1] + delta_emis + gamma_emis - ocean_uptake - land_uptake
        O_s[i] = O_s[i-1] + ocean_uptake - ocean_mixing
        O_d[i] = O_d[i-1] + ocean_mixing
        B[i]   = B[i-1] + 0.6 * land_uptake
        S[i]   = S[i-1] + 0.4 * land_uptake

    return A[-1]

# Binary search
low, high = -0.55, -0.45
best_reduction, best_error = None, float('inf')

for _ in range(100):
    mid = (low + high) / 2
    final_A = run_model_with_reduction(mid)
    error = abs(final_A - target_pgc)

    if error < best_error:
        best_error = error
        best_reduction = mid

    if error <= tolerance:
        break
    elif final_A / 2.13 < 300:
        high = mid
    else:
        low = mid

final_ppm = run_model_with_reduction(best_reduction) / 2.13
print(f"Required annual reduction: {best_reduction:.4f} PgC/yr per year from 2025")
print(f"Atmospheric CO₂ in 2100:   {final_ppm:.1f} ppm (target: 300 ppm)")

## 5. Sensitivity Analysis

Key model uncertainties: ocean uptake rate, land photosynthesis response, land-use change variability.

In [ ]:
def run_sensitivity(ocean_factor=1.0, land_factor=1.0):
    """Run BAU scenario with scaled ocean and land uptake parameters."""
    A, B, S, O_s, O_d = [np.full_like(years_full, val, dtype=float)
                          for val in [A0, B0, S0, Os0, Od0]]

    for i in range(1, len(years_full)):
        year = years_full[i]
        delta_emis = land_use_emissions(year)
        gamma_emis = fossil_emissions(year)

        ocean_uptake = max(0.025 * ocean_factor * (A[i-1] - 600), 0)
        land_uptake  = max(0.020 * land_factor  * (A[i-1] - 600), 0)
        ocean_mixing = 0.015 * (O_s[i-1] - O_d[i-1] / 40)

        A[i]   = A[i-1] + delta_emis + gamma_emis - ocean_uptake - land_uptake
        O_s[i] = O_s[i-1] + ocean_uptake - ocean_mixing
        O_d[i] = O_d[i-1] + ocean_mixing
        B[i]   = B[i-1] + 0.6 * land_uptake
        S[i]   = S[i-1] + 0.4 * land_uptake

    return A

A_base      = run_sensitivity()
A_ocean_hi  = run_sensitivity(ocean_factor=1.15)
A_ocean_lo  = run_sensitivity(ocean_factor=0.85)
A_land_hi   = run_sensitivity(land_factor=1.10)
A_land_lo   = run_sensitivity(land_factor=0.90)

plt.figure(figsize=(12, 6))
plt.plot(years_full, A_base     / 2.13, 'black',  lw=2,   label='Baseline BAU')
plt.fill_between(years_full, A_ocean_lo / 2.13, A_ocean_hi / 2.13,
                 alpha=0.3, color='blue',  label='Ocean uptake ±15%')
plt.fill_between(years_full, A_land_lo  / 2.13, A_land_hi  / 2.13,
                 alpha=0.3, color='green', label='Land uptake ±10%')
plt.xlabel('Year')
plt.ylabel('Atmospheric CO₂ (ppm)')
plt.title('Model Sensitivity Analysis — Key Parameter Uncertainties')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

ocean_range = (A_ocean_hi[-1] - A_ocean_lo[-1]) / 2.13
land_range  = (A_land_hi[-1]  - A_land_lo[-1])  / 2.13
print(f"Ocean uptake sensitivity (±15%): ±{ocean_range/2:.1f} ppm in 2100")
print(f"Land uptake sensitivity (±10%):  ±{land_range/2:.1f} ppm in 2100")